# 🔍 TruthLens - Deep Learning Model Training
## Attention BiLSTM for Fake News Detection

**Team Logic Lords | Heritage Institute of Technology**

---

### Instructions:
1. Upload `Fake.csv` and `True.csv` using the folder icon on the left
2. Run all cells in order
3. Download trained model at the end

In [ ]:
# Check GPU availability
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("\n✅ Setup complete!")

In [ ]:
# Install dependencies
!pip install -q scikit-learn nltk pandas numpy

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
print("✅ Dependencies installed")

In [ ]:
# Preprocessing functions
import re
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

STOP_WORDS = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in STOP_WORDS and len(w) > 2]
    return ' '.join(tokens)

def load_isot_dataset():
    print("[preprocess] Loading ISOT dataset...")
    fake = pd.read_csv('Fake.csv')
    real = pd.read_csv('True.csv')
    
    fake['label'] = 1
    real['label'] = 0
    
    df = pd.concat([fake, real], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"  Cleaning {len(df)} articles...")
    df['content'] = (df['title'] + ' ' + df['text']).apply(clean_text)
    df = df[df['content'].str.len() > 50]
    
    print(f"  Final dataset: {len(df)} articles")
    return df[['content', 'label']]

print("✅ Preprocessing functions loaded")

In [ ]:
# Load and prepare data
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("📊 Loading dataset...")
df = load_isot_dataset()

# Tokenization
MAX_VOCAB = 80000
MAX_LEN = 512

print("🔤 Tokenizing...")
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df['content'])
sequences = tokenizer.texts_to_sequences(df['content'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = df['label'].values

# Train/Val/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

print(f"✅ Data prepared:")
print(f"   Train: {len(X_train)}")
print(f"   Val: {len(X_val)}")
print(f"   Test: {len(X_test)}")
print(f"   Vocab size: {min(MAX_VOCAB, len(tokenizer.word_index))}")

In [ ]:
# Build Attention BiLSTM Model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, Layer
import tensorflow.keras.backend as K

class AttentionLayer(Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], self.units), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True)
        self.u = self.add_weight(shape=(self.units, 1), initializer="glorot_uniform", trainable=True)
        super().build(input_shape)

    def call(self, H):
        score = K.tanh(K.dot(H, self.W) + self.b)
        score = K.dot(score, self.u)
        alpha = K.softmax(score, axis=1)
        context = K.sum(H * alpha, axis=1)
        return context, alpha

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units})
        return cfg

# Hyperparameters
EMB_DIM = 128
CONV_FILTERS = 128
CONV_KERNEL = 5
LSTM_UNITS = 128
ATTN_DIM = 64
DROPOUT = 0.3

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index)) + 1

inp = Input(shape=(MAX_LEN,))
x = Embedding(vocab_size, EMB_DIM, mask_zero=False)(inp)
x = Conv1D(CONV_FILTERS, CONV_KERNEL, activation="relu", padding="same")(x)
x = MaxPooling1D(pool_size=2)(x)
x = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True))(x)
context, _ = AttentionLayer(ATTN_DIM)(x)
x = Dense(128, activation="relu")(context)
x = Dropout(DROPOUT)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(DROPOUT)(x)
out = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inp, outputs=out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])

print("✅ Model built!")
model.summary()

In [ ]:
# Train the model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

BATCH_SIZE = 64
EPOCHS = 15
PATIENCE = 3

callbacks = [
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
    ModelCheckpoint("best_model.keras", save_best_only=True)
]

print("🚀 Training started... (15-30 minutes with GPU)")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("✅ Training complete!")

In [ ]:
# Evaluate the model
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n📊 Test Results:")
print(f"   Accuracy: {acc*100:.2f}%")
print(f"   Loss: {loss:.4f}")

# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Save model and tokenizer
import pickle

model.save('attention_bilstm_model.keras')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("✅ Model saved!")
print("\n📥 Download these files:")
print("   1. attention_bilstm_model.keras")
print("   2. tokenizer.pkl")
print("\nRight-click on files in left sidebar → Download")

In [ ]:
# Test prediction
test_text = "BREAKING: Secret government documents reveal shocking truth!"

def predict_news(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    pred = model.predict(padded, verbose=0)[0][0]
    
    verdict = "FAKE" if pred > 0.5 else "REAL"
    confidence = pred if pred > 0.5 else 1 - pred
    
    print(f"Text: {text}")
    print(f"Verdict: {verdict}")
    print(f"Confidence: {confidence*100:.2f}%")

predict_news(test_text)